# 06b — LSTM Model — Weekly Pure (weekly input, 1-week-ahead)

Uses **weekly-aggregated** data (W-FRI resampling, same as ARIMA/VAR/RF/XGBoost) to
predict the next week's silver log-return. Directly comparable to the other weekly models.

**Limitation**: only ~330 training sequences after aggregation and SEQ_LEN warmup.
Results should be interpreted cautiously — LSTMs typically need more data to generalise.

| Variant | Features |
|---|---|
| LSTM-Y | Weekly silver return only |
| LSTM-EXOG | Silver + weekly market returns |

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings, os, sys
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 120

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: mps


## 1. Hyperparameters

In [9]:
SEQ_LEN  = 20    # lookback: 20 weeks (~5 months)
HORIZON  = 1     # 1 week ahead
HIDDEN   = 32
N_LAYERS = 1
DROPOUT  = 0.2
EPOCHS   = 150
LR       = 1e-3
PATIENCE = 15
BATCH    = 16    # smaller batch — fewer sequences

TARGET = 'silver_return'

## 2. Load & aggregate to weekly

In [10]:
train_d = pd.read_csv('../../data/processed/train.csv', index_col=0, parse_dates=True)
val_d   = pd.read_csv('../../data/processed/val.csv',   index_col=0, parse_dates=True)
test_d  = pd.read_csv('../../data/processed/test.csv',  index_col=0, parse_dates=True)

EXOG = ['gold_return', 'usd_return', 'copper_return', 'sp500_return', 'vix_return', 'oil_return']
FEAT_COLS = [TARGET] + [c for c in EXOG if c in train_d.columns]

# Aggregate: returns sum (additive), sentiment mean
def to_weekly(df):
    return df[FEAT_COLS].resample('W-FRI').sum().dropna()

train = to_weekly(train_d)
val   = to_weekly(val_d)
test  = to_weekly(test_d)

# Merge weekly sentiment if available
sent_path = '../../data/processed/daily_sentiment.csv'
if os.path.exists(sent_path):
    sent = pd.read_csv(sent_path, index_col=0, parse_dates=True)
    sent_w = sent[['reddit_sentiment','news_sentiment']].resample('W-FRI').mean()
    for df in [train, val, test]:
        for col in ['reddit_sentiment','news_sentiment']:
            df[col] = sent_w[col].reindex(df.index).ffill().fillna(0)
    print('Sentiment merged.')
else:
    for df in [train, val, test]:
        df['reddit_sentiment'] = 0.0
        df['news_sentiment']   = 0.0

print(f'Train weeks: {len(train)}  Val weeks: {len(val)}  Test weeks: {len(test)}')
print(f'Features: {train.columns.tolist()}')

Sentiment merged.
Train weeks: 365  Val weeks: 52  Test weeks: 175
Features: ['silver_return', 'gold_return', 'usd_return', 'copper_return', 'sp500_return', 'vix_return', 'oil_return', 'reddit_sentiment', 'news_sentiment']


## 3. Architecture & helpers

In [11]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def make_sequences(data, seq_len, target_col, horizon=1):
    X, y = [], []
    for i in range(seq_len, len(data) - horizon + 1):
        X.append(data[i - seq_len:i])
        y.append(np.sum(data[i:i + horizon, target_col]))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def run_variant(name, feature_cols):
    print(f'\n{"=" * 50}\nVariant: {name}\n{"=" * 50}')
    cols       = [c for c in feature_cols if c in train.columns]
    target_idx = cols.index(TARGET)

    scaler = StandardScaler().fit(train[cols].fillna(0))
    tr_s   = scaler.transform(train[cols].fillna(0))
    va_s   = scaler.transform(val[cols].fillna(0))
    te_s   = scaler.transform(test[cols].fillna(0))

    X_tr, y_tr = make_sequences(tr_s, SEQ_LEN, target_idx, HORIZON)
    X_va, y_va = make_sequences(va_s, SEQ_LEN, target_idx, HORIZON)
    X_te, y_te = make_sequences(te_s, SEQ_LEN, target_idx, HORIZON)
    dates      = test.index[SEQ_LEN:len(test) - HORIZON + 1]

    print(f'  Train seqs: {len(X_tr)}  Val seqs: {len(X_va)}  Test seqs: {len(X_te)}')

    def to_loader(X, y, shuffle=True):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)

    train_loader = to_loader(X_tr, y_tr)
    val_loader   = to_loader(X_va, y_va, shuffle=False)

    ckpt  = f'../../data/processed/lstm_{name.lower().replace("+","_").replace(" ","_")}_pure_weekly_best.pt'
    model = LSTMForecaster(len(cols), HIDDEN, N_LAYERS, DROPOUT).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    crit  = nn.MSELoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

    best_val, pat_cnt = np.inf, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = np.mean([crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
                          for xb, yb in val_loader])
        sched.step(vl)
        if epoch % 25 == 0:
            print(f'  Epoch {epoch:3d}  val={vl:.6f}')
        if vl < best_val:
            best_val, pat_cnt = vl, 0
            torch.save(model.state_dict(), ckpt)
        else:
            pat_cnt += 1
            if pat_cnt >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(torch.load(ckpt))
    model.eval()
    preds_s, acts_s = [], []
    with torch.no_grad():
        test_loader = to_loader(X_te, y_te, shuffle=False)
        for xb, yb in test_loader:
            preds_s.extend(model(xb.to(DEVICE)).cpu().numpy().flatten())
            acts_s.extend(yb.numpy().flatten())

    mu, sigma  = scaler.mean_[target_idx], scaler.scale_[target_idx]
    preds      = np.array(preds_s) * sigma + HORIZON * mu
    actuals    = np.array(acts_s)  * sigma + HORIZON * mu

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    da   = np.mean(np.sign(actuals) == np.sign(preds))
    wda  = np.sum(np.abs(actuals) * (np.sign(actuals) == np.sign(preds))) / np.sum(np.abs(actuals))
    print(f'  RMSE={rmse:.6f}  MAE={mae:.6f}  DA={da:.3f}  WDA={wda:.3f}')
    print(f'  Test sequences: {len(preds)} weekly predictions')

    return {'model': name, 'rmse': rmse, 'mae': mae, 'dir_acc': da, 'wda': wda}, preds, actuals, dates

## 4. Train variants

In [12]:
MARKET_FEATURES = [TARGET] + [c for c in EXOG if c in train.columns]

variants = {
    'LSTM-Y':      [TARGET],
    'LSTM-EXOG':   MARKET_FEATURES,
    'LSTM-REDDIT': MARKET_FEATURES + ['reddit_sentiment'],
    'LSTM-NEWS':   MARKET_FEATURES + ['news_sentiment'],
}

results     = {}
all_preds   = {}
actuals_arr = None
dates_arr   = None

for name, cols in variants.items():
    m, preds, acts, dates = run_variant(name, cols)
    results[name]   = m
    all_preds[name] = preds
    actuals_arr     = acts
    dates_arr       = dates


Variant: LSTM-Y
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 18
  RMSE=0.053494  MAE=0.038249  DA=0.574  WDA=0.593
  Test sequences: 155 weekly predictions

Variant: LSTM-EXOG
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 17
  RMSE=0.053491  MAE=0.038385  DA=0.503  WDA=0.525
  Test sequences: 155 weekly predictions

Variant: LSTM-REDDIT
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 17
  RMSE=0.053975  MAE=0.038640  DA=0.503  WDA=0.519
  Test sequences: 155 weekly predictions

Variant: LSTM-NEWS
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 17
  RMSE=0.053614  MAE=0.038495  DA=0.523  WDA=0.582
  Test sequences: 155 weekly predictions


## 5. Results

In [13]:
metrics_df = pd.DataFrame(list(results.values()))
metrics_df.to_csv('../../data/processed/metrics_lstm_pure_weekly.csv', index=False)

print(f'{"Model":<20}  {"RMSE":>10}  {"MAE":>10}  {"DA":>6}  {"WDA":>6}')
print('-' * 58)
for _, row in metrics_df.iterrows():
    print(f'{row["model"]:<20}  {row["rmse"]:>10.6f}  {row["mae"]:>10.6f}'
          f'  {row["dir_acc"]:>6.3f}  {row["wda"]:>6.3f}')

Model                       RMSE         MAE      DA     WDA
----------------------------------------------------------
LSTM-Y                  0.053494    0.038249   0.574   0.593
LSTM-EXOG               0.053491    0.038385   0.503   0.525
LSTM-REDDIT             0.053975    0.038640   0.503   0.519
LSTM-NEWS               0.053614    0.038495   0.523   0.582


## 6. Period breakdown

In [14]:
sys.path.insert(0, os.path.abspath('../../src'))
from eval_utils import period_metrics, PERIODS

best_name = max(results, key=lambda k: results[k]['wda'])
best_pred = all_preds[best_name]
print('Best variant by WDA:', best_name)

res = period_metrics(actuals_arr, best_pred, dates_arr, PERIODS)
display(res[['n', 'DA', 'WDA']].style
        .format({'n': '{:.0f}', 'DA': '{:.3f}', 'WDA': '{:.3f}'})
        .background_gradient(cmap='RdYlGn', subset=['DA', 'WDA'], vmin=0.35, vmax=0.65))
res[['n', 'DA', 'WDA']].to_csv('../../data/processed/period_lstm_pure_weekly.csv')

Best variant by WDA: LSTM-Y


,n,DA,WDA
Period,,,
2023 (choppy),32,0.531,0.486
2024 (bull start),52,0.500,0.570
2025 (bull run),52,0.692,0.753
2026 (YTD),19,0.526,0.489
── Full test ──,155,0.574,0.593
